# Day 11 — Transfer Learning & Data Augmentation

**Agri-AI Bootcamp** · Phase 3 of 5: *CNNs & Crop Disease* · Days 9–12

> Freeze ResNet18, replace the final layer, apply transforms.v2 and train a Tomato Leaf Disease Classifier.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

A model trained on ImageNet already knows edges, textures and shapes. Freezing the backbone and training only a new final layer needs little data and GPU time. Augmentation creates varied copies of each image so the model copes with real-farm conditions.

### 🌾 Agriculture example

ResNet18 plus a new head for tomato diseases; RandomResizedCrop and ColorJitter imitate different sun angles and phone cameras.

> **Watch out:** Use ImageNet mean/std normalisation, and no random augmentation on validation images.

## 2. Setup

In [ ]:
!pip -q install torch torchvision --upgrade 2>/dev/null | tail -1

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torchvision import models
from torchvision.transforms import v2
from torch.utils.data import TensorDataset, DataLoader
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)
print("device:", dev)

## 3. Data

In [ ]:
# ---- Leaf image helper -------------------------------------------------------
# Real course dataset: PlantVillage / Tomato leaf images on Kaggle (needs a
# kaggle.json API token).  So that this notebook runs anywhere, we fall back to
# a small SYNTHETIC leaf set: green patches = "healthy", green patches with
# brown lesions = "diseased".  Swap in the real folder when you have it.
import numpy as np, torch

def synth_leaves(n=400, size=64, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, 3, size, size), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        leaf = np.stack([rng.uniform(.10, .25, (size, size)),
                         rng.uniform(.45, .70, (size, size)),
                         rng.uniform(.10, .25, (size, size))])
        diseased = i % 2
        if diseased:                      # add 3-6 brown lesions
            for _ in range(rng.integers(3, 7)):
                cx, cy = rng.integers(8, size - 8, 2)
                r = rng.integers(4, 9)
                yy, xx = np.ogrid[:size, :size]
                m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
                leaf[0][m], leaf[1][m], leaf[2][m] = .45, .28, .10
        X[i] = leaf
        y[i] = diseased
    return torch.tensor(X), torch.tensor(y)

CLASSES = ["healthy", "diseased"]
X_img, y_img = synth_leaves()
print("images:", tuple(X_img.shape), "| labels:", torch.bincount(y_img).tolist())

# ImageNet statistics -- a pre-trained backbone expects exactly this normalisation
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

train_aug = v2.Compose([                       # random ONLY for training
    v2.RandomResizedCrop(224, scale=(0.6, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(0.3, 0.3, 0.3),             # different sun angles / phone cameras
    v2.Normalize(IMNET_MEAN.flatten().tolist(), IMNET_STD.flatten().tolist())])
eval_tf = v2.Compose([                         # deterministic for validation
    v2.Resize((224, 224), antialias=True),
    v2.Normalize(IMNET_MEAN.flatten().tolist(), IMNET_STD.flatten().tolist())])

perm = torch.randperm(len(y_img)); X_img, y_img = X_img[perm], y_img[perm]
cut = int(.8*len(y_img))
Xtr, ytr, Xte, yte = X_img[:cut], y_img[:cut], X_img[cut:], y_img[cut:]
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from torchvision import models
from torchvision.transforms import v2
m=models.resnet18(weights="DEFAULT")
for p in m.parameters(): p.requires_grad=False
m.fc=nn.Linear(512,10)   # tomato disease classes
aug=v2.Compose([v2.RandomResizedCrop(224),v2.ColorJitter(.3,.3,.3),
                v2.ToImage(),v2.ToDtype(torch.float32,scale=True)])
```

## 5. Worked solution

In [ ]:
# ---- Freeze the ImageNet backbone, train only a new head --------------------
try:
    net = models.resnet18(weights="DEFAULT")
    pretrained = True
except Exception as e:                      # offline runtime
    print("could not download weights, falling back to random init:", str(e)[:70])
    net = models.resnet18(weights=None); pretrained = False

for p in net.parameters(): p.requires_grad = False       # freeze everything
net.fc = nn.Linear(512, len(CLASSES))                    # new, trainable head
net = net.to(dev)

trainable = [n for n, p in net.named_parameters() if p.requires_grad]
print(f"pretrained={pretrained} | trainable tensors: {trainable} "
      f"({sum(p.numel() for p in net.parameters() if p.requires_grad):,} of "
      f"{sum(p.numel() for p in net.parameters()):,} params)")

def batches(X, y, bs=32, aug=None, shuffle=True):
    idx = torch.randperm(len(y)) if shuffle else torch.arange(len(y))
    for i in range(0, len(idx), bs):
        j = idx[i:i+bs]
        xb = X[j]
        xb = aug(xb) if aug is not None else eval_tf(xb)
        yield xb.to(dev), y[j].to(dev)

opt   = torch.optim.Adam(net.fc.parameters(), 1e-3)      # only the head
lossf = nn.CrossEntropyLoss()
for epoch in range(5):
    net.train()
    for xb, yb in batches(Xtr, ytr, aug=train_aug):
        opt.zero_grad(); lossf(net(xb), yb).backward(); opt.step()
    net.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in batches(Xte, yte, aug=None, shuffle=False):
            c += (net(xb).argmax(1) == yb).sum().item(); t += len(yb)
    print(f"epoch {epoch} test acc {c/t:.3f}  (head only)")

# ---- Stage 2: unfreeze the last block and fine-tune at a SMALL lr ------------
for p in net.layer4.parameters(): p.requires_grad = True
opt = torch.optim.Adam([{"params": net.fc.parameters(),     "lr": 1e-3},
                        {"params": net.layer4.parameters(), "lr": 1e-4}])  # 10x smaller
for epoch in range(3):
    net.train()
    for xb, yb in batches(Xtr, ytr, aug=train_aug):
        opt.zero_grad(); lossf(net(xb), yb).backward(); opt.step()
    net.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in batches(Xte, yte, aug=None, shuffle=False):
            c += (net(xb).argmax(1) == yb).sum().item(); t += len(yb)
    print(f"fine-tune epoch {epoch} test acc {c/t:.3f}")

# ---- See what augmentation does to one image --------------------------------
def unnorm(t): return (t.cpu()*IMNET_STD + IMNET_MEAN).clamp(0,1).permute(1,2,0)
fig, ax = plt.subplots(1, 6, figsize=(13, 2.3))
ax[0].imshow(X_img[0].permute(1,2,0)); ax[0].set_title("original", fontsize=8); ax[0].axis("off")
for i in range(5):
    ax[i+1].imshow(unnorm(train_aug(X_img[0:1])[0])); ax[i+1].axis("off")
    ax[i+1].set_title(f"aug {i+1}", fontsize=8)
plt.suptitle("one leaf, five training views -- validation gets none of this", fontsize=10)
plt.show()

## 6. Your turn

1. Train the same head WITHOUT augmentation and compare the train/test gap.
2. Unfreeze the whole backbone at lr=1e-3 and watch it get worse — that is catastrophic forgetting.
3. Point this at the real Kaggle tomato set (10 classes) and change only `nn.Linear(512, 10)`.

In [ ]:
# TODO 1: no augmentation
# ... reuse the loop above with aug=eval_tf and compare

# TODO 2: unfreeze all at a big lr
# for p in net.parameters(): p.requires_grad = True
# opt = torch.optim.Adam(net.parameters(), 1e-3)   # expect the accuracy to fall

# TODO 3: real tomato data (10 classes)
# !kaggle datasets download -d kaustubhb999/tomatoleaf -q && unzip -q tomatoleaf.zip
# from torchvision.datasets import ImageFolder
# ds = ImageFolder("tomato/train", transform=train_aug)
# net.fc = nn.Linear(512, len(ds.classes))

## 7. Dataset and references

**Dataset:** [Kaggle: Tomato leaf disease images](https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf)

**References**

- [PyTorch Transfer Learning](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

**Course material:** PyTorch Transfer Learning Tutorial

---
**Next:** Day 12 — Imbalanced Data & Advanced Metrics